library

In [2]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

 GENERATE SIMPLE IMAGE DATASET

In [3]:
# 16x16 grayscale images

N = 600
img_size = 16

X = np.random.rand(N, img_size, img_size)

# create simple labels (circle-like pattern)

y = ((np.sum(X[:,4:12,4:12],axis=(1,2))) > 32).astype(int).reshape(-1,1)


TRAIN / VAL / TEST SPLIT

In [4]:
idx = np.random.permutation(N)

train_end = int(0.7*N)
val_end = int(0.85*N)

train_idx = idx[:train_end]
val_idx = idx[train_end:val_end]
test_idx = idx[val_end:]

X_train,y_train = X[train_idx],y[train_idx]
X_val,y_val = X[val_idx],y[val_idx]
X_test,y_test = X[test_idx],y[test_idx]


CONVOLUTION FUNCTION

In [5]:
def conv2d(image, kernel):

    k = kernel.shape[0]
    h,w = image.shape

    out_size = h - k + 1
    output = np.zeros((out_size,out_size))

    for i in range(out_size):
        for j in range(out_size):

            region = image[i:i+k, j:j+k]
            output[i,j] = np.sum(region * kernel)

    return output

MAX POOLING

In [6]:
def maxpool(feature_map, size=2):

    h,w = feature_map.shape
    out = np.zeros((h//size, w//size))

    for i in range(0,h,size):
        for j in range(0,w,size):

            region = feature_map[i:i+size, j:j+size]
            out[i//size, j//size] = np.max(region)

    return out


 ACTIVATION

In [7]:
def relu(z):
    return np.maximum(0,z)

def sigmoid(z):
    z = np.clip(z,-500,500)
    return 1/(1+np.exp(-z))

BCE LOSS

In [8]:
def BCE(y,yhat):

    eps = 1e-8
    yhat = np.clip(yhat,eps,1-eps)

    return -np.mean(y*np.log(yhat)+(1-y)*np.log(1-yhat))

INITIALIZE PARAMETERS

In [9]:
kernel = np.random.randn(3,3) * 0.01

dense_w = np.random.randn(49,1) * 0.01
dense_b = 0


FORWARD PASS

In [10]:

def forward(img):

    conv = conv2d(img,kernel)

    act = relu(conv)

    pool = maxpool(act)

    flat = pool.flatten().reshape(1,-1)

    z = flat @ dense_w + dense_b

    yhat = sigmoid(z)

    cache = (img,conv,act,pool,flat)

    return yhat,cache


TRAINING LOOP

In [11]:



lr = 0.01
epochs = 30

for epoch in range(epochs):

    losses = []

    for i in range(len(X_train)):

        img = X_train[i]
        target = y_train[i]

        yhat,cache = forward(img)

        loss = BCE(target,yhat)
        losses.append(loss)

        img,conv,act,pool,flat = cache

        # gradients dense layer

        dZ = yhat - target

        dW = flat.T @ dZ
        db = dZ

        # update dense

        dense_w -= lr * dW
        dense_b -= lr * db

    if epoch % 5 == 0:
        print("Epoch",epoch,"Loss",np.mean(losses))



Epoch 0 Loss 0.692273401560397
Epoch 5 Loss 0.6909959183038971
Epoch 10 Loss 0.6909893605912835
Epoch 15 Loss 0.6909826186814457
Epoch 20 Loss 0.6909758760007715
Epoch 25 Loss 0.6909691337125065


TEST ACCURACY

In [12]:
correct = 0

for i in range(len(X_test)):

    pred,_ = forward(X_test[i])

    if (pred >= 0.5) == y_test[i]:
        correct += 1

acc = correct / len(X_test)

print("\nTest Accuracy:",acc)


Test Accuracy: 0.43333333333333335
